In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from ydata_profiling import ProfileReport
import pickle

pd.options.display.max_columns = None
pd.options.display.max_colwidth = 50

data_folder = "../data/"
images_folder = "../data/images/"
pipelines_folder = "../pipelines/"
df_total = pd.read_csv(os.path.join(data_folder, 'items_phase_1.csv'))
df_train = pd.read_csv(os.path.join(data_folder, 'items_train.csv'))
df_task_1 = pd.read_csv(os.path.join(data_folder, 'task_1.csv'))

# Notebook `items_phase_1.csv`

In [2]:
# profile = ProfileReport(df_total, title="items_phase_1", explorative=True)
# profile.to_notebook_iframe()

In [3]:
print("Length of dataset:", len(df_total))

Length of dataset: 199835


In [4]:
df_total.sample(2)

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo
31102,1305056,24.0,6449,['42'],NaN,Trendyol Blue 2-Piece Boy Hooded Jacket-Pants ...,With its regular mold design that offers comfo...,sk
47386,1318209,165.0,"778,1453057",['1'],NaN,Portafoglio grande da donna Furla,Portafoglio grande da donna Furla Camelia WP00...,it


## Key takeaway
- missingy se musi resit u:
    - description (3%)
    - brandEditionTagId (99.8%) - to je asi target?
    - colorTagIdsString (3.1%)
---
# Notebook `task_1.csv`
- kazdy radek je jedna skupina se sloupci item - item4(to jsou id do items_phase_1.csv - itemId)
- Kazdy 

In [5]:
print("Length of dataset:", len(df_task_1))

Length of dataset: 15000


In [6]:
# profile = ProfileReport(df_task_1, title="task_1", explorative=True)
# profile.to_notebook_iframe()

In [7]:
df_task_1.head()

,item1,item2,item3,item4,item5
0,130622,292253,463442,483968,1253745
1,82627,388496,553738,638400,884327
2,46130,333489,644448,848154,1178149
3,150796,248537,742067,1206230,1280786
4,76610,196894,345145,620255,932761


---
# Dataset `items_train.csv`
- obsahuje toho min, vim ze ma stejny hodnoty v departmentIds


In [8]:
print("Total records:", len(df_train))
print("Total null values:\n", df_train.isnull().sum())

Total records: 928234
Total null values:
 itemId                    0
price                     0
colorTagIdsString     27834
departmentIds             0
brandEditionTagId    925518
title                     0
description           35473
geo                       0
label                     0
dtype: int64


In [9]:
df_train.sample(2)

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo,label
493494,516378,31040.00,231,['1'],NaN,Sportcipők Asics,Asics Sportcipők Lyte Classic1202A306 Fehér,hu,90859
299291,694254,47.99,231,['1'],NaN,Calvin Klein Jeans Bluza STACKED INSTITUTIONAL...,"Material: 100% bombaž, Barva: bela , Koda proi...",si,64598


In [10]:
# profile = ProfileReport(df_train, title="task_1", explorative=True)
# profile.to_notebook_iframe()


---
# Vycisteni dat 
- prevod na spolecnou menu 
- normalizace meny v danem geo uzemi
- doplneni null hodnot
- null ve sloupci `colorTagIdString` muzu nahradit 0 
- `colorTagIdString` a `departmentIds` je potreba roztrhnout - obsahuji vice hodnot oddelenych carkou

## Tvorba preprocessing pipeline

In [11]:
df_total[df_total["brandEditionTagId"] == 0]

,itemId,price,colorTagIdsString,departmentIds,brandEditionTagId,title,description,geo


In [12]:
df_train.columns

Index(['itemId', 'price', 'colorTagIdsString', 'departmentIds',
       'brandEditionTagId', 'title', 'description', 'geo', 'label'],
      dtype='object')

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from PriceGeoTransformer import PriceGeoTransformer
from DepartmentIdsTransformer import DepartmentIdsCleaner
import numpy as np

imput_zero_cols = ['colorTagIdsString', "brandEditionTagId"]
input_zero_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
])

input_unknown_cols = ["description"]
input_unknown_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value="Unknown")),
])

categorical_features = ['geo',"price"]
categorical_transformer = Pipeline(steps=[
    ('PriceGeoTransformer', PriceGeoTransformer())
])


# prevadi na stejny format jako jsou barvy - cisla oddelena carkou 
department_features = ['departmentIds']
department_transformer = Pipeline(steps=[
    ('DepartmentIdsCleaner', DepartmentIdsCleaner())
])

# Combine preprocessing for numeric and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('zero', input_zero_transformer, imput_zero_cols),
        ('unknown', input_unknown_transformer, input_unknown_cols),
        ('geo', categorical_transformer, categorical_features),
        ('department', department_transformer, department_features)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

preprocessor.set_output(transform="pandas")



pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

X_train = df_train.drop(columns=['label'])
y_train = df_train['label']

X_train_transformed = pipeline.fit_transform(X_train)
df_total_transformed = pipeline.transform(df_total)

In [14]:
pickle.dump(pipeline, open(os.path.join(pipelines_folder, 'preprocessing_pipeline.pkl'), 'wb'))

## Příprava pro PyTorch dataset
- PyTorch bude použit na vytvoření embeddingů sloupců s více kategorijema a na kategorické sloupce
- PyTorch umí totiž lépe tvořit řídké matice
- Vytvoříme si mappingy pro jednotlivé kategorie...

In [18]:
from GlamiDatasetVocabulary import GlamiVocabularyManager
vocab_manager = GlamiVocabularyManager()
vocab_manager.fit(df_total_transformed)
vocab_manager.save()

## V tomto stavu:
- Nejsou null hodnoty ve sloupcích:
    - colorTagIdString
    - description
    -  brandEditionTagId
- Barvy a deptId maji stejny format - cisla oddelena carkou
- chybi udelat OneHot encodingy kategorickejch - to udelame v PyTorchi